PERSONAL FINANCE ANALYZER

### What I'm doing in this project
1. I'm loading and inspecting my bank transactions.
2. I'm classifying the different transaction types.
3. I'm cleaning up the merchant names.
4. I'm organizing my purchases into categories.
5. I'm calculating a summary of my finances.
6. I'm analyzing how much I spent in each category.
7. I'm creating charts to visualize my spending patterns.


## 1. Setting Up and Loading My Data

I'm starting by importing the libraries I'll use throughout the project. Then I'm loading my CSV file and converting the `Date` column from text into a Pandas datetime value so I can work with the dates correctly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load bank transactions
df = pd.read_csv("data/sample_transactions.csv")

# Convert dates so Pandas can sort, group, and analyze them correctly
df["Date"] = pd.to_datetime(df["Date"])

df.head()

### Inspecting my dataset

Before I make any changes, I'm checking the size of the dataset, its column names, and its data types. This helps me confirm that the file loaded correctly.

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

## 2. Classifying My Transaction Types

My raw bank statement includes purchases, incoming money, transfers, and debt payments. I'm creating a `Transaction_Type` column so I can separate and analyze each type of transaction.


In [ ]:
conditions = [
    df["Description"].str.contains("Zelle Payment From", case=False, na=False),
    df["Description"].str.contains("Zelle Payment To", case=False, na=False),
    df["Description"].str.contains("Capital One|Discover E-Payment", case=False, na=False),
    df["Description"].str.contains("Payment Sent.*Chime", case=False, na=False, regex=True),
]

choices = [
    "Money In",
    "Transfer Out",
    "Debt Payment",
    "Transfer Out",
]

# Anything that does not match the rules is treated as a normal purchase
df["Transaction_Type"] = np.select(
    conditions,
    choices,
    default="Purchase"
)

df["Transaction_Type"].value_counts()

In [ ]:
# Quick check of the classification
df[["Date", "Description", "Amount", "Transaction_Type"]].head(20)

## 3. Cleaning the Merchant Names

The merchant descriptions in my bank statement are a little messy, so I'm cleaning them up to make them easier to understand. I'm keeping all of my cleanup rules together and applying them with a loop instead of creating a separate cell for every merchant.


In [ ]:
# Start every transaction as "Other" until a merchant rule matches it
df["Merchant"] = "Other"

merchant_rules = {
    # Food / transportation / travel
    "Uber": r"Uber",
    "DoorDash": r"DoorDash",
    "United Airlines": r"United",
    "PJ's Coffee": r"Pj's Coffee",
    "Raising Cane's": r"Raising Canes",

    # Shopping / entertainment / car
    "Amazon": r"Amazon",
    "Walmart": r"Wm Supercenter",
    "AutoZone": r"Autozone",
    "Fandango": r"Fandango",
    "Gazebo Smoke Shop": r"Gazebo Smoke Shop",

    # Gas / convenience
    "Delta Mini Mart": r"Delta Mini Mart",
    "Exxon": r"Exxon",
    "Shell": r"Shell Oil",
    "Chevron": r"Chevron",
    "CircleK": r"Circle\s*K",
    "QT": r"\bQT\b",
    "Liquor Store": r"Liquor Store",

    # Recurring / digital services
    "Spotify": r"Spotify",
    "Prime Video Channels": r"Prime Video Channels",
    "Rocket Money": r"Rocket Money Premium",
    "Apple": r"Apple\.Com",
    "LA Farm Bureau": r"LA Farm Bureau",
    "Mint Mobile": r"Mint Mobile",
    "Rocketfast Car Wash": r"Rocketfast",

    # Financing / fitness
    "Affirm": r"Affirm",
    "Lambright LaTech Gym": r"Lambright",

    # Financial movements
    "Capital One": r"Capital One",
    "Discover": r"Discover",
    "Chime": r"Chime",
    "Zelle": r"Zelle",
}

for merchant, pattern in merchant_rules.items():
    matches = df["Description"].str.contains(
        pattern,
        case=False,
        na=False,
        regex=True
    )
    df.loc[matches, "Merchant"] = merchant

# Most common cleaned merchant names
df["Merchant"].value_counts()

### Reviewing unmatched merchants

Now I'm checking the transactions that are still labeled `Other`. This gives me a chance to spot any merchants that may need an additional cleanup rule.

In [ ]:
other_merchants = df.loc[
    df["Merchant"] == "Other",
    ["Description", "Amount"]
]

other_merchants

## 4. Categorizing My Spending

At this point, the `Merchant` column shows me **where** my money went. Now I'm adding a `Category` column to show me **what type of spending** each purchase represents.


In [ ]:
category_map = {
    # Travel / food
    "United Airlines": "Travel",
    "DoorDash": "Food Delivery",
    "PJ's Coffee": "Dining",
    "Raising Cane's": "Dining",

    # Transportation / gas
    "Uber": "Transportation",
    "Exxon": "Gas/Convenience",
    "Delta Mini Mart": "Gas/Convenience",
    "QT": "Gas/Convenience",
    "Shell": "Gas/Convenience",
    "CircleK": "Gas/Convenience",
    "Chevron": "Gas/Convenience",
    "Liquor Store": "Gas/Convenience",

    # Shopping / groceries / car
    "Amazon": "Shopping",
    "Walmart": "Groceries",
    "AutoZone": "Car",
    "Rocketfast Car Wash": "Car",

    # Bills / subscriptions / digital
    "LA Farm Bureau": "Insurance",
    "Mint Mobile": "Phone",
    "Fandango": "Entertainment",
    "Spotify": "Subscription",
    "Prime Video Channels": "Subscription",
    "Rocket Money": "Subscription",
    "Apple": "Digital Services",

    # Financing / fitness
    "Affirm": "Financing",
    "Lambright LaTech Gym": "Fitness",
}

# Map known merchants to categories and label unmatched merchants as Other
df["Category"] = df["Merchant"].map(category_map).fillna("Other")

# Financial movements are not normal purchase categories
df.loc[df["Transaction_Type"] == "Money In", "Category"] = "Money In"
df.loc[df["Transaction_Type"] == "Transfer Out", "Category"] = "Transfer"
df.loc[df["Transaction_Type"] == "Debt Payment", "Category"] = "Debt Payment"

df[["Date", "Merchant", "Amount", "Transaction_Type", "Category"]].head(20)

### Checking uncategorized purchases

I'm checking only my actual purchases for anything that is still uncategorized because the financial transfers already have their own categories.

In [ ]:
uncategorized_purchases = df.loc[
    (df["Transaction_Type"] == "Purchase") & (df["Category"] == "Other"),
    ["Merchant", "Description", "Amount"]
]

uncategorized_purchases

## 5. Creating My Financial Summary

I'm creating a separate `purchases` DataFrame so my purchase analysis doesn't accidentally include transfers or debt payments.

I'm using `.copy()` so this becomes an independent DataFrame that I can safely modify later.

In [ ]:
purchases = df.loc[
    df["Transaction_Type"] == "Purchase"
].copy()

money_in = df.loc[df["Transaction_Type"] == "Money In", "Amount"].sum()
total_spending = purchases["Amount"].abs().sum()
debt_payments = df.loc[df["Transaction_Type"] == "Debt Payment", "Amount"].abs().sum()
transfers_out = df.loc[df["Transaction_Type"] == "Transfer Out", "Amount"].abs().sum()
net_cash_flow = df["Amount"].sum()

summary_report = pd.DataFrame({
    "Metric": [
        "Money In",
        "Purchase Spending",
        "Debt Payments",
        "Transfers Out",
        "Net Cash Flow",
    ],
    "Amount": [
        money_in,
        total_spending,
        debt_payments,
        transfers_out,
        net_cash_flow,
    ],
})

summary_report["Amount"] = summary_report["Amount"].round(2)
summary_report

## 6. Analyzing My Spending by Category

Now I'm grouping my purchases by category and calculating how many dollars I spent in each one. I'm also calculating the percentage each category represents out of my total purchase spending.

In [ ]:
category_spending = (
    purchases
    .groupby("Category")["Amount"]
    .sum()
    .abs()
    .sort_values(ascending=False)
)

category_percent = (category_spending / total_spending) * 100

category_report = pd.DataFrame({
    "Amount": category_spending.round(2),
    "Percent": category_percent.round(2),
})

category_report

## 7. Visualizing My Results

I'm using a bar chart here because it makes it easier for me to compare my spending across the different categories.

In [ ]:
category_spending.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Spending by Category")
plt.xlabel("Category")
plt.ylabel("Amount Spent ($)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Looking at my daily spending over time

Since I converted the `Date` column to a datetime value at the beginning, I can now group my purchases by day and plot them as a time series. This helps me see how my spending changed throughout the month.

In [ ]:
daily_spending = (
    purchases
    .groupby("Date")["Amount"]
    .sum()
    .abs()
)

daily_spending.plot(
    kind="line",
    figsize=(12, 6),
    marker="o"
)

plt.title("Daily Spending Over Time")
plt.xlabel("Date")
plt.ylabel("Amount Spent ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Reviewing My Final Clean Dataset

I'm finishing with an analysis-ready table that keeps the original transaction description and adds the cleaned merchant name, transaction type, and spending category.

In [ ]:
final_columns = [
    "Date",
    "Description",
    "Merchant",
    "Amount",
    "Transaction_Type",
    "Category",
]

clean_transactions = df[final_columns].copy()
clean_transactions.head(20)

## What I'm Demonstrating in This Project

- I'm loading and inspecting CSV data with Pandas.
- I'm cleaning dates and text-based transaction descriptions.
- I'm creating classifications with NumPy conditions.
- I'm using `.loc[]` to update selected rows.
- I'm mapping merchants to categories with dictionaries and `.map()`.
- I'm filtering DataFrames with `.copy()`.
- I'm grouping and summarizing financial data.
- I'm building summary tables.
- I'm creating basic visualizations with Matplotlib.
